# A ParamRefs DynamicMap design pattern

**This seems to be the way forward!** 

Still to do: 

1) [x] insert actual hyperspectral image cube and highres tif panes
2) [x] insert ROI spectral curves overlay
3) [x] sync labels with boxes and spectra
4) [ ] Add NdLayout for spectra 
5) [ ] combine all reactivity in a Viewer 
6) [ ] load initial ROI's from TOML file and update edits   

In [ ]:
from ipynb_path import get
get()

## Data now

In [ ]:
from fairdatanow import data_now

import holoviews as hv 
from holoviews import RGB, opts 
from holoviews.operation.datashader import rasterize
import panel as pn 
import numpy as np 

In [ ]:
hv.extension('bokeh') 
opts.defaults(opts.RGB(aspect='equal', frame_width=300, fontsize=10), 
             opts.Labels(text_color='white', fontsize=8), 
             opts.Rectangles(fill_color=None, line_color='white')) 

In [ ]:
url = 'https://laboppad.nl/ukiyo-e-world' 

In [ ]:
toml_txt = '''
# On which locations within our ten World Museum Japanse prints did Gauthier and Tessa measure an XRF spectrum? 
# And what do these spectra show?  

# Measurements have been done on a selection of 10 prints that are present in our lab 
# here are the 10 object numbers of th prints located in Amsterdam

object_numbers = [
 'RV-1-4468-544',
 'RV-1-4469-484',
 'RV-1-4469-58',
 'RV-1-4469-x6',
 'RV-1-4469Q',
 'RV-1-4470-12',
 'RV-1-4470-27',
 'RV-360-2345g',
 'RV-360-2359-2',
 'RV-360-6886']


# The data table is linked to all search queries necessary for fairdatanow within this table are subtables that indicate the type of file the querie downloads.
# These subtables don't necessarily have to be file typing but can also be grouped by object. For this example we chose to use filetype to lessen the amount of branching at the top of the dictionary generated by the data_now function.
[data.spx]
# these are regexes that download all .spx files per object number  
RV-1-4468-544 = ".*XRF.*RV-1-4468-544.*[.]spx"
RV-1-4469-484 = ".*XRF.*RV-1-4469-484.*[.]spx"
RV-1-4469-58 = ".*XRF.*RV-1-4469-58.*[.]spx"
RV-1-4469-x6 = ".*XRF.*RV-1-4469-x6.*[.]spx"
RV-1-4469Q = ".*XRF.*RV-1-4469Q.*[.]spx"       # duplicate so no XRF spectra 
RV-1-4470-12 = ".*XRF.*RV-1-4470-12.*[.]spx"
RV-1-4470-27 = ".*XRF.*RV-1-4470-27.*[.]spx"
RV-360-2345g = ".*XRF.*RV-360-2345g.*[.]spx"
RV-360-2359-2 = ".*XRF.*RV-360-2359-2.*[.]spx"
RV-360-6886 = ".*XRF.*RV-360-6886.*[.]spx"

[data.jpg]
# these are regexes that download all .jpg files per object number  
RV-1-4468-544 = ".*XRF.*RV-1-4468-544.*[.]jpg"
RV-1-4469-484 = ".*XRF.*RV-1-4469-484.*[.]jpg"
RV-1-4469-58 = ".*XRF.*RV-1-4469-58.*[.]jpg"
RV-1-4469-x6 = ".*XRF.*RV-1-4469-x6.*[.]jpg"
RV-1-4469Q = ".*XRF.*RV-1-4469Q.*[.]jpg" # duplicate so no jpg's for point locations 
RV-1-4470-12 = ".*XRF.*RV-1-4470-12.*[.]jpg"
RV-1-4470-27 = ".*XRF.*RV-1-4470-27.*[.]jpg"
RV-360-2345g = ".*XRF.*RV-360-2345g.*[.]jpg"
RV-360-2359-2 = ".*XRF.*RV-360-2359-2.*[.]jpg"
RV-360-6886 = ".*XRF.*RV-360-6886.*[.]jpg"

[data.tif] 
# these are regexes that download all .tif files per object number 
RV-1-4468-544 = ".*akama.*RV-1-4468-544[.]tif"
RV-1-4469-484 = ".*akama.*RV-1-4469-484[.]tif"
RV-1-4469-58 = ".*akama.*RV-1-4469-58[.]tif"
RV-1-4469-x6 = ".*akama.*RV-1-4469-x6[.]tif"
RV-1-4469Q = ".*akama.*RV-1-4469Q[.]tif"
RV-1-4470-12 = ".*akama.*RV-1-4470-12[.]tif"
RV-1-4470-27 = ".*akama.*1-4470-27[.]tif"      # TIF NAME WITHOUT RV prefix!  
RV-360-2345g = ".*akama.*RV-360-2345-?g[.]tif"
RV-360-2359-2 = ".*akama.*RV-360-2359-2[.]tif"
RV-360-6886 = ".*akama.*RV-360-6886[.]tif" 

# here are the 10 corresponding spectral data cubes processed by Gauthier and Tessa 
[data.npz] 
RV-1-4468-544 = ".*RIS/interim/.*RV-1-4468-544.*[.]npz"
RV-1-4469-484 = ".*RIS/interim/.*RV-1-4469-484.*[.]npz"
RV-1-4469-58 = ".*RIS/interim/.*RV-1-4469-58.*[.]npz"
RV-1-4469-x6 = ".*RIS/interim/.*RV-1-4469-x6.*[.]npz"
RV-1-4469Q = ".*RIS/interim/.*RV-1-4469Q.*[.]npz"
RV-1-4470-12 = ".*RIS/interim/.*RV-1-4470-12.*[.]npz"
RV-1-4470-27 = ".*RIS/interim/.*RV-1-4470-27.*[.]npz"
RV-360-2345g = ".*RIS/interim/.*RV-360-2345g.*[.]npz"
RV-360-2359-2 = ".*RIS/interim/.*RV-360-2359-2.*[.]npz"
RV-360-6886 = ".*RIS/interim/.*RV-360-6886.*[.]npz"
'''

In [ ]:
data = data_now(url, toml_txt)

In [ ]:
rgb_views = [rasterize(RGB.load_image(files[0]).opts(title=f'[{i}] {num}')) for i, [num, files] in enumerate(data['tif'].items())] 
layout = hv.Layout(rgb_views).opts(shared_axes=False).cols(5)
layout

Let's pick a number

In [ ]:
obj_nums = list(data['npz'].keys())
print(obj_nums)

In [ ]:
n = 6
num = obj_nums[n]
npz = np.load(data['npz'][num][0])
cube = npz['image'][:,:, ::-1].transpose(1, 2, 0)
wavelengths = npz['wavelengths'] 
h, w, d = cube.shape
bounds = [0, 0, w, h]
tif_file = data['tif'][num][0]
pseudo_rgb = cube[:,:, [70, 53, 19]] 

## TODO 4: NdLayout for spectra base on parametrized callbacks (TILL HERE)

Let's look up again how the datashader [Advanced Stockexplorer](https://holoviews.org/user_guide/Dashboards.html#replacing-the-output) dashboard works. I realize now that the DynamicMap is returned as a method within a parametrized function.

In [ ]:
import holoviews as hv
from holoviews.streams import BoxEdit
import param 
import panel as pn
import re 
import numpy as np
hv.extension('bokeh') 

class ROI_editor(param.Parameterized): 

    def view(self): 

        

## TODO 4: NdLayout for spectra (not working) 

Can not get this code to work beyond generating white views. 

Let's look up an NdLayout example

https://holoviews.org/reference/containers/bokeh/NdLayout.html

In [ ]:
import holoviews as hv
from holoviews.streams import BoxEdit
import param 
import panel as pn
import re 
import numpy as np
hv.extension('bokeh') 

pn.extension('codeeditor')

# this code is based on these two tutorials: 
# https://param.holoviz.org/en/docs/latest/user_guide/Dependencies_and_Watchers.html#watchers
# https://panel.holoviz.org/reference/widgets/CodeEditor.html


class LabelEditor(param.Parameterized):
    lines = param.String(default='# Hi there\n') 

    def __init__(self, **params): 
        super().__init__(**params) 
        self.param.watch(self.add_newline, ['lines'], queued=True)

    def add_newline(self, event):  
        if (not self.lines.endswith('\n')): 
            self.lines = f'{self.lines}\n'

    def editor_widget(self): 
        title = pn.pane.Str('ROI Labels')
        editor = pn.widgets.CodeEditor.from_param(self.param.lines, max_height=100, on_keyup=False) # avoids jumping editor 
        return pn.Column(title, editor)
    

editor = LabelEditor()

# dynamic map callback to create reactive labels 
def roi_labelize(roi_data, label_lines): 
    '''Add labels to boxes stream `data` and return both as a holoviews Overlay.'''
    
    if (roi_data is None) or (len(roi_data['x0']) == 0):
        labels = hv.Labels([])
    else: 
        # append newlines to match at least the number of ROI's 
        n_lines = label_lines.count('\n') 
        n_rois = len(roi_data['x0']) 
        if n_rois > n_lines: 
            n_extra = n_rois - n_lines 
            label_lines = label_lines + n_extra * '# \n'
            
        # split multi line string
        lines_list = re.split('\n', label_lines) 
        
        # simple hack to position labels 
        label_list = [[xi, yi, f'#{i+1} {lines_list[i]}'] for i, [xi, yi] in enumerate(zip(roi_data['x1'], roi_data['y0']))]   
        labels = hv.Labels(label_list).opts(text_align='left')

    return labels 


# dynamic map callback to create labeled reactive roi spectra 
def roi_spectra(roi_data, label_lines): 

    print('hi from callback')

    lines_list = re.split('\n', label_lines) 

    if (roi_data is None): # or (len(roi_data['x0']) == 0):
        print(f'no ROI spectral data: {roi_data}') 
        xx0 = np.arange(400, 1000) 
        yy0 = np.ones_like(xx0) 
        yy1 = 0.5 * np.ones_like(xx0) 
        curve0 = hv.Curve((xx0, yy0), kdims='Wavelength', vdims='Intensity') 
        curve1 = hv.Curve((xx0, yy1), kdims='Wavelength', vdims='Intensity') 
        curve_dict = {0: curve0, 1: curve1} 
        layout = hv.NdLayout(curve_dict, kdims='ROI') 
        
    else: 
        print(f'Found spectral data: {roi_data}')
        curve_dict = {} 
        rois = zip(roi_data['x0'], roi_data['x1'], roi_data['y0'], roi_data['y1'])
        for i, (x0, x1, y0, y1) in enumerate(rois):
            selection = ds.select(x=(x0, x1), y=(y0, y1))
            try: 
                label = lines_list[i]
            except: 
                label = '-'  

            # perhaps something is wrong here so let's comment this line out and simply repeat the flatlines code above? 
            curve = hv.Curve(selection.aggregate('Wavelength', np.mean), kdims='ROI').opts(framewise=True)
            
            #curve_dict[f'#{i+1} {label}'] = curve # should use label_lines
            curve_dict[i] = curve
        layout = hv.NdLayout(curve_dict, kdims='ROI') 

    return layout


# create a cube Dataset  
h, w, d = cube.shape 
ds = hv.Dataset((wavelengths, np.arange(w), np.arange(h-1, -1, -1), cube), ['Wavelength', 'x', 'y'], 'Reflectance') 

# high res and pseudo rgb views 
tif_view = rasterize(RGB.load_image(tif_file, bounds=bounds))
pseudo_view = hv.RGB(pseudo_rgb, bounds=bounds) 

# regions of interest stream 
boxes_canvas = hv.Rectangles([]) 
boxes_stream = BoxEdit(source=boxes_canvas)
refs = {'roi_data': boxes_stream.param.data, 'label_lines': editor.param.lines} # later we can add the code editor reference here 
streams = [hv.streams.ParamRefs(refs=refs)]

# create dynamic maps 
labels_dmap = hv.DynamicMap(roi_labelize, streams=streams)#.opts(frame_width=frame_width)
spectra_dmap = hv.DynamicMap(roi_spectra, streams=streams) 

# combine all four panes 
viewer = pn.Row(tif_view, pseudo_view * boxes_canvas * labels_dmap, spectra_dmap, editor.editor_widget)

viewer

## TODO 3: Sync labels (works)

In [ ]:
import holoviews as hv
from holoviews.streams import BoxEdit
import param 
import panel as pn
import re 
hv.extension('bokeh') 

pn.extension('codeeditor')

# this code is based on these two tutorials: 
# https://param.holoviz.org/en/docs/latest/user_guide/Dependencies_and_Watchers.html#watchers
# https://panel.holoviz.org/reference/widgets/CodeEditor.html


class LabelEditor(param.Parameterized):
    lines = param.String(default='# Hi there\n') 

    def __init__(self, **params): 
        super().__init__(**params) 
        self.param.watch(self.add_newline, ['lines'], queued=True)

    def add_newline(self, event):  
        if (not self.lines.endswith('\n')): 
            self.lines = f'{self.lines}\n'

    def editor_widget(self): 
        title = pn.pane.Str('ROI Labels')
        editor = pn.widgets.CodeEditor.from_param(self.param.lines, max_height=100, on_keyup=False) # avoids jumping editor 
        return pn.Column(title, editor)
    

editor = LabelEditor()

# dynamic map callback to create reactive labels 
def roi_labelize(roi_data, label_lines): 
    '''Add labels to boxes stream `data` and return both as a holoviews Overlay.'''
    
    if (roi_data is None) or (len(roi_data['x0']) == 0):
        labels = hv.Labels([])
    else: 
        # append newlines to match at least the number of ROI's 
        n_lines = label_lines.count('\n') 
        n_rois = len(roi_data['x0']) 
        if n_rois > n_lines: 
            n_extra = n_rois - n_lines 
            label_lines = label_lines + n_extra * '# \n'
            
        # split multi line string
        lines_list = re.split('\n', label_lines) 
        
        # simple hack to position labels 
        label_list = [[xi, yi, f'#{i+1} {lines_list[i]}'] for i, [xi, yi] in enumerate(zip(roi_data['x1'], roi_data['y0']))]   
        labels = hv.Labels(label_list).opts(text_align='left')

    return labels 


# dynamic map callback to create labeled reactive roi spectra 
def roi_spectra(roi_data, label_lines): 

    print('Hello from roi_spectra callback')
    lines_list = re.split('\n', label_lines) 

    if (roi_data is None) or (len(roi_data['x0']) == 0):
        spectrum_view = hv.NdOverlay({'abc': hv.Curve([], kdims='Wavelength', vdims='Reflectance')}, )
    else: 
        curves = {}
        rois = zip(roi_data['x0'], roi_data['x1'], roi_data['y0'], roi_data['y1'])
        for i, (x0, x1, y0, y1) in enumerate(rois):
            selection = ds.select(x=(x0, x1), y=(y0, y1))
            try: 
                label = lines_list[i]
            except: 
                label = '-' 
            curves[f'#{i+1} {label}'] = hv.Curve(selection.aggregate('Wavelength', np.mean), kdims='Wavelength').opts(framewise=True) # RATHER CRUCIAL HERE!!! 
            spectrum_view = hv.NdOverlay(curves, kdims='ROI spectra').opts(legend_position='right', legend_offset=(10, 200)) 

    return spectrum_view


# create a cube Dataset  
h, w, d = cube.shape 
ds = hv.Dataset((wavelengths, np.arange(w), np.arange(h-1, -1, -1), cube), ['Wavelength', 'x', 'y'], 'Reflectance') 

# high res and pseudo rgb views 
tif_view = rasterize(RGB.load_image(tif_file, bounds=bounds))
pseudo_view = hv.RGB(pseudo_rgb, bounds=bounds) 

# regions of interest stream 
boxes_canvas = hv.Rectangles([]) 
boxes_stream = BoxEdit(source=boxes_canvas)
refs = {'roi_data': boxes_stream.param.data, 'label_lines': editor.param.lines} # later we can add the code editor reference here 
streams = [hv.streams.ParamRefs(refs=refs)]

# create dynamic maps 
labels_dmap = hv.DynamicMap(roi_labelize, streams=streams)#.opts(frame_width=frame_width)
spectra_dmap = hv.DynamicMap(roi_spectra, streams=streams).opts(frame_width=500) 

# combine all four panes 
viewer = pn.Row(tif_view, pseudo_view * boxes_canvas * labels_dmap, spectra_dmap, editor.editor_widget)

viewer

Is it possible to watch the boxes stream and adjust the number of lines accordingly?  

## TODO's 1+2: insert cube panes and add spectral curves (works)

In [ ]:
import holoviews as hv
from holoviews.streams import BoxEdit
import param 
import panel as pn
import re 
hv.extension('bokeh') 

pn.extension('codeeditor')

# this code is based on these two tutorials: 
# https://param.holoviz.org/en/docs/latest/user_guide/Dependencies_and_Watchers.html#watchers
# https://panel.holoviz.org/reference/widgets/CodeEditor.html

class LabelEditor(param.Parameterized):
    lines = param.String(default='# Hi there\n') 

    def __init__(self, **params): 
        super().__init__(**params) 
        self.param.watch(self.add_newline, ['lines'], queued=True)

    def add_newline(self, event):  
        if (not self.lines.endswith('\n')): 
            self.lines = f'{self.lines}\n'

    def editor_widget(self): 
        title = pn.pane.Str('Annotations')
        editor = pn.widgets.CodeEditor.from_param(self.param.lines, max_height=100, on_keyup=False) # avoids jumping editor 
        return pn.Column(title, editor)
    

editor = LabelEditor()

# dynamic map callback to create reactive labels 
def roi_labelize(roi_data, label_lines): 
    '''Add labels to boxes stream `data` and return both as a holoviews Overlay.'''
    
    if (roi_data is None) or (len(roi_data['x0']) == 0):
        labels = hv.Labels([])
    else: 
        # append newlines to match at least the number of ROI's 
        n_lines = label_lines.count('\n') 
        n_rois = len(roi_data['x0']) 
        if n_rois > n_lines: 
            n_extra = n_rois - n_lines 
            label_lines = label_lines + n_extra * '# \n'
            
        # split multi line string
        lines_list = re.split('\n', label_lines) 
        
        # simple hack to position labels 
        label_list = [[xi, yi, f' {lines_list[i]}'] for i, [xi, yi] in enumerate(zip(roi_data['x1'], roi_data['y0']))]   
        labels = hv.Labels(label_list).opts(text_align='left')

    return labels 


# dynamic map callback to create labeled reactive roi spectra 
def roi_spectra(roi_data, label_lines): 

    if (roi_data is None) or (len(roi_data['x0']) == 0):
        spectrum_view = hv.NdOverlay({'abc': hv.Curve([], kdims='Wavelength', vdims='Reflectance')}, )
    else: 
        curves = {}
        rois = zip(roi_data['x0'], roi_data['x1'], roi_data['y0'], roi_data['y1'])
        for i, (x0, x1, y0, y1) in enumerate(rois):
            selection = ds.select(x=(x0, x1), y=(y0, y1))
            curves[f'#{i}'] = hv.Curve(selection.aggregate('Wavelength', np.mean), kdims='Wavelength').opts(framewise=True) # should use label_lines
            spectrum_view = hv.NdOverlay(curves, kdims='ROI spectra') 

    return spectrum_view

    


# create a cube Dataset  
h, w, d = cube.shape 
ds = hv.Dataset((wavelengths, np.arange(w), np.arange(h-1, -1, -1), cube), ['Wavelength', 'x', 'y'], 'Reflectance') 

# high res and pseudo rgb views 
tif_view = rasterize(RGB.load_image(tif_file, bounds=bounds))
pseudo_view = hv.RGB(pseudo_rgb, bounds=bounds) 

# regions of interest stream 
boxes_canvas = hv.Rectangles([]) 
boxes_stream = BoxEdit(source=boxes_canvas)
refs = {'roi_data': boxes_stream.param.data, 'label_lines': editor.param.lines} # later we can add the code editor reference here 
streams = [hv.streams.ParamRefs(refs=refs)]

# create dynamic maps 
labels_dmap = hv.DynamicMap(roi_labelize, streams=streams)#.opts(frame_width=frame_width)
spectra_dmap = hv.DynamicMap(roi_spectra, streams=streams) 

# combine all four panes 
viewer = pn.Row(tif_view, pseudo_view * boxes_canvas * labels_dmap, spectra_dmap, editor.editor_widget)

viewer

## A ParamRefs DynamicMap ROI editor 

This works! Also fixed the jumping code editor with CTRL-Enter. 

In [ ]:
import holoviews as hv
from holoviews.streams import BoxEdit
import param 
import panel as pn
import re 
hv.extension('bokeh') 

pn.extension('codeeditor')

# this code is based on these two tutorials: 
# https://param.holoviz.org/en/docs/latest/user_guide/Dependencies_and_Watchers.html#watchers
# https://panel.holoviz.org/reference/widgets/CodeEditor.html

class LabelEditor(param.Parameterized):
    lines = param.String(default='# Hi there\n') 

    def __init__(self, **params): 
        super().__init__(**params) 
        self.param.watch(self.add_newline, ['lines'], queued=True)

    def add_newline(self, event):  
        if (not self.lines.endswith('\n')): 
            self.lines = f'{self.lines}\n'

    def editor_widget(self): 
        title = pn.pane.Str('Annotations')
        editor = pn.widgets.CodeEditor.from_param(self.param.lines, max_height=100, on_keyup=False) # avoids jumping editor 
        return pn.Column(title, editor)
    

editor = LabelEditor()


def roi_labelize(roi_data, label_lines): 
    '''Add labels to boxes stream `data` and return both as a holoviews Overlay.'''
    
    if (roi_data is None) or (len(roi_data['x0']) == 0):
        labels = hv.Labels([])
    else: 
        # append newlines to match at least the number of ROI's 
        n_lines = label_lines.count('\n') 
        n_rois = len(roi_data['x0']) 
        if n_rois > n_lines: 
            n_extra = n_rois - n_lines 
            label_lines = label_lines + n_extra * '# \n'
            
        # split multi line string
        lines_list = re.split('\n', label_lines) 
        
        # simple hack to position labels 
        label_list = [[xi, yi, f' {lines_list[i]}'] for i, [xi, yi] in enumerate(zip(roi_data['x1'], roi_data['y0']))]   
        labels = hv.Labels(label_list).opts(text_align='left')

    return labels 

boxes_canvas = hv.Rectangles([]) 
boxes_stream = BoxEdit(source=boxes_canvas)
refs = {'roi_data': boxes_stream.param.data, 'label_lines': editor.param.lines} # later we can add the code editor reference here 
streams = [hv.streams.ParamRefs(refs=refs)]

# create interactive plots 
#spectra_dmap = hv.DynamicMap(roi_spectra, streams=[box_stream]).opts(frame_width=frame_width, framewise=True)

labels_dmap = hv.DynamicMap(roi_labelize, streams=streams)#.opts(frame_width=frame_width)


viewer = pn.Row(boxes_canvas * labels_dmap, editor.editor_widget)

viewer

## A simple `DynamicMap` labelizer  

See hyperspectral roi picker notebook cell 26. Here is the minimal example for dynamic labels. This approach makes direct use of the `boxes_stream` in a `DynamicMap` callback. 

In [ ]:
import holoviews as hv
from holoviews.streams import BoxEdit
import param 
hv.extension('bokeh') 

labelz = ['A', 'B', 'C'] 

def roi_labelize(data): 
    '''Add labels to boxes stream `data` and return both as a holoviews Overlay.'''
    
    if (data is None) or (len(data['x0']) == 0):
        labels = hv.Labels([])
    else: 
        # simple hack to position labels 
        label_list = [[xi, yi, f'    {labelz[i]}'] for i, [xi, yi] in enumerate(zip(data['x1'], data['y0']))]   
        labels = hv.Labels(label_list)

    return labels 

boxes_canvas = hv.Rectangles([]) 
boxes_stream = BoxEdit(source=boxes_canvas)

# create interactive plots 
#spectra_dmap = hv.DynamicMap(roi_spectra, streams=[box_stream]).opts(frame_width=frame_width, framewise=True)

labels_dmap = hv.DynamicMap(roi_labelize, streams=[boxes_stream])#.opts(frame_width=frame_width)

#rgb_view = rasterize(RGB(pseudo_rgb, bounds=bounds)).opts(frame_width=frame_width) 


In [ ]:
boxes_canvas * labels_dmap

Ok, this works for three boxes! If we add a fourth (which induces a mistake because we have no label 'D'), we interestingly get a completely white plot **WITHOUT ANY WARNING**!

## Feeding the `roi_labelize` callback with a `ParamRefs` stream 

Preparing for the `Viewer` design pattern we need all our components to be reactive. Therefore we need to upgrade the simple holoviews stream to a ParamRefs stream.  

In [ ]:
import holoviews as hv
from holoviews.streams import BoxEdit
import param 
hv.extension('bokeh') 

labelz = ['A', 'B', 'C'] 

def roi_labelize(roi_data): 
    '''Add labels to boxes stream `data` and return both as a holoviews Overlay.'''
    
    if (roi_data is None) or (len(roi_data['x0']) == 0):
        labels = hv.Labels([])
    else: 
        # simple hack to position labels 
        label_list = [[xi, yi, f'    {labelz[i]}'] for i, [xi, yi] in enumerate(zip(roi_data['x1'], roi_data['y0']))]   
        labels = hv.Labels(label_list)

    return labels 

boxes_canvas = hv.Rectangles([]) 
boxes_stream = BoxEdit(source=boxes_canvas)
refs = {'roi_data': boxes_stream.param.data} # later we can add the code editor reference here 
streams = [hv.streams.ParamRefs(refs=refs)]

# create interactive plots 
#spectra_dmap = hv.DynamicMap(roi_spectra, streams=[box_stream]).opts(frame_width=frame_width, framewise=True)

labels_dmap = hv.DynamicMap(roi_labelize, streams=streams)#.opts(frame_width=frame_width)


In [ ]:
boxes_canvas * labels_dmap

## Zipping in the CodeEditor 

First add the code editor pane, then combine the streams. 

In [ ]:
import holoviews as hv
from holoviews.streams import BoxEdit
import param 
import panel as pn
import re 
hv.extension('bokeh') 

pn.extension('codeeditor')

# this code is based on these two tutorials: 
# https://param.holoviz.org/en/docs/latest/user_guide/Dependencies_and_Watchers.html#watchers
# https://panel.holoviz.org/reference/widgets/CodeEditor.html

class LabelEditor(param.Parameterized):
    lines = param.String(default='# Hi there\n') 

    def __init__(self, **params): 
        super().__init__(**params) 
        self.param.watch(self.add_newline, ['lines'], queued=True)

    def add_newline(self, event):  
        if (not self.lines.endswith('\n')): 
            self.lines = f'{self.lines}\n'

    def editor_widget(self): 
        title = pn.pane.Str('Annotations')
        editor = pn.widgets.CodeEditor.from_param(self.param.lines, max_height=100)
        return pn.Column(title, editor)
    

editor = LabelEditor()

labelz = ['A', 'B', 'C'] 

def roi_labelize(roi_data, label_lines): 
    '''Add labels to boxes stream `data` and return both as a holoviews Overlay.'''
    
    if (roi_data is None) or (len(roi_data['x0']) == 0):
        labels = hv.Labels([])
    else: 
        # append newlines to match at least the number of ROI's 
        n_lines = label_lines.count('\n') 
        n_rois = len(roi_data['x0']) 
        if n_rois > n_lines: 
            n_extra = n_rois - n_lines 
            label_lines = label_lines + n_extra * '# \n'
            
        # split multi line string
        lines_list = re.split('\n', label_lines) 
        
        # simple hack to position labels 
        label_list = [[xi, yi, f'    {lines_list[i]}'] for i, [xi, yi] in enumerate(zip(roi_data['x1'], roi_data['y0']))]   
        labels = hv.Labels(label_list)

    return labels 

boxes_canvas = hv.Rectangles([]) 
boxes_stream = BoxEdit(source=boxes_canvas)
refs = {'roi_data': boxes_stream.param.data, 'label_lines': editor.param.lines} # later we can add the code editor reference here 
streams = [hv.streams.ParamRefs(refs=refs)]

# create interactive plots 
#spectra_dmap = hv.DynamicMap(roi_spectra, streams=[box_stream]).opts(frame_width=frame_width, framewise=True)

labels_dmap = hv.DynamicMap(roi_labelize, streams=streams)#.opts(frame_width=frame_width)


viewer = pn.Row(boxes_canvas * labels_dmap, editor.editor_widget)

viewer

## How to fix the jumping code editor? 

And ill positioned labels. 

In [ ]:
import holoviews as hv
from holoviews.streams import BoxEdit
import param 
import panel as pn
import re 
hv.extension('bokeh') 

pn.extension('codeeditor')

# this code is based on these two tutorials: 
# https://param.holoviz.org/en/docs/latest/user_guide/Dependencies_and_Watchers.html#watchers
# https://panel.holoviz.org/reference/widgets/CodeEditor.html

class LabelEditor(param.Parameterized):
    lines = param.String(default='# Hi there\n') 

    def __init__(self, **params): 
        super().__init__(**params) 
        self.param.watch(self.add_newline, ['lines'], queued=True)

    def add_newline(self, event):  
        if (not self.lines.endswith('\n')): 
            self.lines = f'{self.lines}\n'

    def editor_widget(self): 
        title = pn.pane.Str('Annotations')
        editor = pn.widgets.CodeEditor.from_param(self.param.lines, max_height=100, on_keyup=False) # avoids jumping editor 
        return pn.Column(title, editor)
    

editor = LabelEditor()

labelz = ['A', 'B', 'C'] 

def roi_labelize(roi_data, label_lines): 
    '''Add labels to boxes stream `data` and return both as a holoviews Overlay.'''
    
    if (roi_data is None) or (len(roi_data['x0']) == 0):
        labels = hv.Labels([])
    else: 
        # append newlines to match at least the number of ROI's 
        n_lines = label_lines.count('\n') 
        n_rois = len(roi_data['x0']) 
        if n_rois > n_lines: 
            n_extra = n_rois - n_lines 
            label_lines = label_lines + n_extra * '# \n'
            
        # split multi line string
        lines_list = re.split('\n', label_lines) 
        
        # simple hack to position labels 
        label_list = [[xi, yi, f' {lines_list[i]}'] for i, [xi, yi] in enumerate(zip(roi_data['x1'], roi_data['y0']))]   
        labels = hv.Labels(label_list).opts(text_align='left')

    return labels 

boxes_canvas = hv.Rectangles([]) 
boxes_stream = BoxEdit(source=boxes_canvas)
refs = {'roi_data': boxes_stream.param.data, 'label_lines': editor.param.lines} # later we can add the code editor reference here 
streams = [hv.streams.ParamRefs(refs=refs)]

# create interactive plots 
#spectra_dmap = hv.DynamicMap(roi_spectra, streams=[box_stream]).opts(frame_width=frame_width, framewise=True)

labels_dmap = hv.DynamicMap(roi_labelize, streams=streams)#.opts(frame_width=frame_width)


viewer = pn.Row(boxes_canvas * labels_dmap, editor.editor_widget)

viewer

## A Parametrized CodeEditor

First the LabelEditor: 

In [ ]:
import panel as pn
import param

pn.extension('codeeditor')

# this code is based on these two tutorials: 
# https://param.holoviz.org/en/docs/latest/user_guide/Dependencies_and_Watchers.html#watchers
# https://panel.holoviz.org/reference/widgets/CodeEditor.html

class LabelEditor(param.Parameterized):
    lines = param.String(default='# Hi there') 

    def __init__(self, **params): 
        super().__init__(**params) 
        self.param.watch(self.add_newline, ['lines'], queued=True)

    def add_newline(self, event):  
        if (not self.lines.endswith('\n')): 
            self.lines = f'{self.lines}\n'

    def editor_widget(self): 
        title = pn.pane.Str('Annotations')
        editor = pn.widgets.CodeEditor.from_param(self.param.lines, max_height=100)
        return pn.Column(title, editor)
    

editor = LabelEditor()

In [ ]:
editor.editor_widget()

## Fusing together with `ParamRefs`

See: https://holoviews.org/user_guide/Responding_to_Events.html#using-parameterized-classes-as-a-stream

Next challenge is to feed the labels to a DynamicMap with the code editor. That should be possible with a `Parametrized` function or with multiple streams using `ParamRefs`. Let's look up the [Declarative Dashboards](https://holoviews.org/user_guide/Dashboards.html#declarative-dashboards) example from the holoviews documentation.  

If all works well we should be able to push the `.lines` string to a stream. Otherwise we should also be able to update the value of `.lines`. 

In [ ]:
editor.lines = '# Hi there \nB\n'

Seems best to combine both the `boxes_stream` parameter and the 'editor` parameter into refs: https://holoviews.org/user_guide/Responding_to_Events.html#what-is-a-parameter-reference

In [ ]:
labels_stream['lines']

## Front end pushing from param (forget it)

Here is a minimal example of a `BoxEdit` stream: 

In [ ]:
import holoviews as hv
from holoviews.streams import BoxEdit
import param 
hv.extension('bokeh')

In [ ]:
boxes = hv.Rectangles([], ['x0', 'x1', 'y0', 'y1']).opts(active_tools=['box_edit'])
boxes_stream = BoxEdit(source=boxes, data={'x0': [0], 'y0':[0], 'x1': [0.5], 'y1':[0.4]}, linked=True)

In [ ]:
boxes

In [ ]:
boxes_stream.data # first draw otherwise this dictionary is None 

In [ ]:
boxes_stream.data = {'x0': [0], 'y0':[0], 'x1': [0.5], 'y1':[0.4]}

In [ ]:
boxes_stream.update(data={'x0': [0], 'y0':[0], 'x1': [0.5], 'y1':[0.4]})

In [ ]:
boxes_stream.element

As you can see, the newly drawn boxes added to the boxes_stream instance. 

Question is now if we could also **push data to** the `boxes_stream` plot from another *parameter*?

In [ ]:
my_boxes_dict = param.Dict(default={'x0': [0], 'y0': [0], 'x1': [0.5], 'y1':[0.4]})

In [ ]:
my_boxes_dict.default

How to update this value anyway? Ok, simply like this: 

In [ ]:
my_boxes_dict = {'x0': [0], 'y0': [0], 'x1': [0.5], 'y1': [0.6]}

In [ ]:
my_boxes_dict.items()

Let's now try to insert this into the `reactive_boxes_stream`:

In [ ]:
boxes_canvas = hv.Rectangles([])

In [ ]:
reactive_boxes_stream = BoxEdit(source=boxes_canvas)

In [ ]:
boxes_canvas

In [ ]:
reactive_boxes_stream.data

Mm, although it seems possible to set the `data` parameter, it does not show in the `boxes_canvas' plot, and is overwritten. Let's try to fiddle a bit. Perhaps this post provides some info: https://github.com/holoviz/holoviews/issues/2978

In [ ]:
from bokeh.models.sources import ColumnDataSource

In [ ]:
ds = ColumnDataSource({'x0': [0], 'y0': [0], 'x1': [0.5], 'y1':[0.4]})

In [ ]:
ds.stream()

In [ ]:
reactive_boxes_stream_with_data = BoxEdit(source=boxes_canvas, data=ds.data)

In [ ]:
boxes_canvas.

The holoviews documentation https://holoviews.org/user_guide/Streaming_Data.html#pipe says this: 

A `Pipe` allows data to be pushed into a `DynamicMap` callback to change a visualization, just like the streams in the Responding to Events user guide were used to push changes to metadata that controlled the visualization. A `Pipe` can be used to push data of any type and make it available to a `DynamicMap callback`. Since all `Element` types accept `data` of various forms we can use `Pipe` to push data directly to the constructor of an Element through a `DynamicMap`.

**My take away from this is that we might be forced to use a `DynamicMap`.** Like this rotating vectors example or the parametrized callback:

In [ ]:
import time

import numpy as np
import pandas as pd

import holoviews as hv
from holoviews import opts
from holoviews.streams import Buffer, Pipe

hv.extension('bokeh')

In [ ]:
pipe = Pipe(data=[])
vector_dmap = hv.DynamicMap(hv.VectorField, streams=[pipe])
vector_dmap.opts(color='Magnitude', xlim=(-1, 1), ylim=(-1, 1))

In [ ]:
x,y  = np.mgrid[-10:11,-10:11] * 0.1
sine_rings  = np.sin(x**2+y**2)*np.pi+np.pi
exp_falloff = 1/np.exp((x**2+y**2)/8)

for i in np.linspace(0, 10, 250):
    time.sleep(0.1)
    pipe.send((x,y,sine_rings*i, exp_falloff))